<a href="https://colab.research.google.com/github/takatakamanbou/AdvML/blob/2025/AdvML2025_ex11notebookA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AdvML ex11notebookA

<img width=72 src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/AdvML/AdvML-logo.png"> [この授業のウェブページ](https://www-tlab.math.ryukoku.ac.jp/wiki/?AdvML)




板書や口頭で補足する前提なので，この notebook だけでは説明が不完全です．


In [ ]:
# 準備あれこれ
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn
seaborn.set_theme()

---
## クラスタリング（前回の続き）
---

前回，非階層型クラスタリング手法の一例として，GMMを用いる例を取り上げた．ここでは，もう一つの例として，K-means アルゴリズムを取り上げる．また，最後に少しだけ他のクラスタリング手法も紹介する．

---
### K-means アルゴリズム

**$K$-means アルゴリズム**（K-平均法， K-means method）は，データ同士の間のユークリッド距離が計算できるデータを対象としたクラスタリングのアルゴリズム．
データをいくつのクラスタに分けるかを予め決めておいて，各データをどれか一つのクラスタに割り振る．
名前の$K$は予め定めるクラスタの数を表す．

例えば，下図左のような2次元のデータに対して，クラスタ数を $K=3$ として$K$-meansアルゴリズムを適用すると，下図右のような結果が得られる．

<img src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/ML/kmeans.png">

右の図の各データ点は，3つのクラスタのうちのどれに割り振られたかに応じて3色に塗り分けられている．
図中に描かれた3つの★印は，それぞれのクラスタに所属するデータの重心を表している．
$K$-meansアルゴリズムでは，クラスタごとのデータの重心のことを **セントロイド**(centroid)と呼ぶ．

［**K-means アルゴリズム**］

$N$個の$D$次元ベクトルから成るデータ集合
$$
\{ \pmb{x}_n \in \mathbb{R}^{D} | n = 1, 2, \ldots, N\}
$$
を学習データとして，これを$K$個のクラスタ $C_1, C_2, \ldots, C_K$ に分ける $K$-means アルゴリズムは，次のようになる．

(0) クラスタごとのセントロイド $\pmb{c}_{k} \in \mathbb{R}^{D}$ ($k=1, 2, \ldots, K$)の初期値を適当に決める．

(1) 学習データ $\pmb{x}_n$ ($n = 1, 2, \ldots, N$) のそれぞれを，次の手順で $C_1, C_2, \ldots, C_K$ のいずれかに割り当てる．
- $\pmb{x}_n$ に対して，$K$個のセントロイドのうち最も距離の小さいものの番号 $y_n$ を次式のように求める．
$$
\newcommand{\argmin}{\mathop{\rm argmin}\limits}
y_n = \argmin_{k=1,2,\ldots,K}\Vert \pmb{x}_{n} - \pmb{c}_{k} \Vert^2
$$
- $\pmb{x}_n$ をクラスタ $C_{y_n}$ に割り当てる

(2) 各クラスタに割り振られた学習データたちの重心を求め，その値でそれぞれのセントロイドを更新する．式で書くと次の通り（注）．
$$
\pmb{c}_{k} = \frac{1}{|C_k|}\sum_{n:y_n = k} \pmb{x}_n
$$

(3) 結果が一定の条件を満たしていれば終了，さもなくば (1) へ戻る．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※ 注: この式の和は，$y_n$ が $k$ と等しいような $n$ たちについてとる．すなわち，クラスタ $C_k$ に所属するデータの和を意味する．また，$|C_k|$ は集合 $C_k$ の元の数，つまり，クラスタ $C_k$ に割り振られたデータの個数を表す．
</span>



$K$-means アルゴリズムの手続きは，GMMの学習手続きと類似している．(1) は E-step に，(2) は M-step に対応していると考えることができる．実際，GMM においてパラメータに制約を加えると，$K$-means アルゴリズムを導くことができる．

$K$-means アルゴリズムには次のような性質がある．

性質1: $K$-means アルゴリズムの手続きでは，次式で表される量が最小化される．
$$
E = \sum_{k=1}^{K}\sum_{n:y_n = k} \Vert \pmb{x}_n - \pmb{c}_{k}\Vert^2
$$
この値は，「学習データのそれぞれが割り振られたクラスタのセントロイドとの距離の二乗」の和である．アルゴリズムの繰り返しごとに，この $E$ の値は単調減少する．

性質2: $K$-means アルゴリズムの学習結果は，初期値のとり方によって変わる．初期値によっては，$E$ の（最小でない）極小解に到達してそれ以上解が変化しなくなる．そのため，実用の際は，何通りかの初期値で学習を繰り返し，$E$の値が最も小さかった結果を採用する，といった方法がとられる．

---
### 実験: 2次元データに K-means アルゴリズムを適用してみる

#### 準備

データと K-means アルゴリズムの学習手続きの関数の定義．

In [ ]:
# 実験用データの入手
df = pd.read_csv('https://www-tlab.math.ryukoku.ac.jp/~takataka/course/ML/data4kmeans.csv')
dat1 = df[['x1', 'x2']].to_numpy()
dat2 = df[['y1', 'y2']].to_numpy()

In [ ]:
## セントロイドの初期化
#
def initCentroid(X, centroid, seed=None):
    assert X.shape[1] == centroid.shape[1]
    K = centroid.shape[0]
    # 学習データからランダムに K 個を選択して初期セントロイドとする
    N = X.shape[0]
    idx = np.arange(N, dtype=int)
    if seed is not None:
        np.random.seed(seed)
    np.random.shuffle(idx)
    centroid[:] = X[idx[:K], :]

## データをクラスタに割り振る
#
def assignCluster(X, centroid, label):
    assert X.shape[1] == centroid.shape[1] and X.shape[0] == label.shape[0]
    K = centroid.shape[0]
    N = X.shape[0]
    sqe = 0.0
    for n in range(N):
        # 各セントロイドとの距離の二乗を計算
        d = np.sum((X[n, :] - centroid)**2, axis=1)
        # 距離最小のクラスタへ割り振る
        i = np.argmin(d)
        label[n] = i
        sqe += d[i]

    return sqe/N  # 割り振られたセントロイドとの距離の二乗の平均

## セントロイドを計算し直す
#
def updateCentroid(X, centroid, label):
    assert X.shape[1] == centroid.shape[1] and X.shape[0] == label.shape[0]
    K = centroid.shape[0]
    for ik in range(K):
        # ik 番目のクラスタに割り当てられたデータの平均をそのクラスタの新しいセントロイドとする
        centroid[ik, :] = np.mean(X[label==ik, :], axis=0)

#### データその1

次のコードセルをそのまま実行すると，`dat1` という変数に格納された2次元データの散布図を描く．

In [ ]:
Xcl = dat1  #　データその1
#Xcl = dat2  # データその2

# クラスタリング結果描画用のデータ
xmin, xmax = -5, 5
ymin, ymax = -5, 5
p = np.dstack(np.mgrid[xmin:xmax:0.05, ymin:ymax:0.05])
P = p.reshape((-1, p.shape[2]))

# 散布図
fig, ax = plt.subplots(facecolor="white", figsize=(4, 4))
ax.scatter(Xcl[:, 0], Xcl[:, 1], s=8)
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect('equal')
plt.show()

次のコードセルは，K-means アルゴリズムの初期化の手続きである．

In [ ]:
## 初期化

K = 3  # クラスタ数

N, D = Xcl.shape
centroid = np.empty((K, D)) # セントロイド
label = np.empty(N, dtype=int) # 各学習データの所属するセントロイドの番号
labelP = np.empty(P.shape[0], dtype=int)

initCentroid(Xcl, centroid)  # セントロイドを初期化

i = 0  # K-平均法の繰り返し回数

上のセルを実行して初期化を行ったのち，下のセルを実行すると，K-meansアルゴリズムの繰り返しの1回分を実行することができる．
繰り返し実行して，学習が進む様子を観察しよう．

In [ ]:
## 1ステップの学習を実行

# データを各クラスタに割り振る
msqe = assignCluster(Xcl, centroid, label)
assignCluster(P, centroid, labelP)

# 現在のクラスタ割り振り結果を描画
colors = seaborn.color_palette(n_colors=K)
fig = plt.figure(facecolor="white", figsize=(10, 5))
ax0 = fig.add_subplot(121)
ax0.set_xlim(xmin, xmax)
ax0.set_ylim(ymin, ymax)
ax0.set_aspect('equal')
for ik in range(K):
    Xk = Xcl[label==ik, :]
    Pk = P[labelP==ik, :]
    ax0.scatter(Xk[:, 0], Xk[:, 1], color=colors[ik], s=8)
    ax0.plot(centroid[ik, 0], centroid[ik, 1], color='white', marker='*', markerfacecolor=colors[ik], markersize=25)
    ax0.scatter(Pk[:, 0], Pk[:, 1], marker='.', alpha=0.1, color=colors[ik])
    ax0.text(xmin+0.5, ymin+0.5, f'step{i}: {msqe:.3f}', size=20)

# セントロイドを更新
updateCentroid(Xcl, centroid, label)
i += 1

# 新しいセントロイドを表示
ax1 = fig.add_subplot(122)
ax1.set_xlim(xmin, xmax)
ax1.set_ylim(ymin, ymax)
ax1.set_aspect('equal')
for ik in range(K):
    Xk = Xcl[label==ik, :]
    ax1.scatter(Xk[:, 0], Xk[:, 1], color=colors[ik], s=8)
    ax1.plot(centroid[ik, 0], centroid[ik, 1], color='white', marker='*', markerfacecolor=colors[ik], markersize=25)

plt.show()

K-means アルゴリズムは，パラメータの初期値（初期セントロイドの選び方）によって学習結果が変わる．初期化からやり直してみよう．

さらに，K の値を変えて実験してみよう．

#### データその2


「データその1」の最初のコードセルの最初の2行のコメントを付け替えることで，`dat2` という変数に入ったデータでも実験することができる．

#### 参考

2024年度「機械学習II」ex11 ( https://www-tlab.math.ryukoku.ac.jp/wiki/?ML/2024#ex11 ) notebookC に，K-means アルゴリズムを猫の顔画像に適用する実験や，カラー画像の画素値をクラスタリングして画像を減色する実験などの例があります．

---
### その他のクラスタリング手法

クラスタリングの手法は，この授業で取り上げたものの他にも多数ある．例えば，scikit-learn User Guide の「2.3. Clustering」のページはこちら:
https://scikit-learn.org/stable/modules/clustering.html

例えば...
- DBSCAN: 「密度の高いところをクラスタとみなす」という考え方に基づいている．クラスタ数をあらかじめ指定する必要がない．
- Spectral Clustering: データ点間の関係をグラフ構造でとらえる．K-means アルゴリズムなどではうまくクラスタリングできないようなデータの構造をとらえることができる．

---
## 次元削減
---




次元削減の目的は，与えられた高次元データを，そこに含まれる本質的な情報をなるべく損なわないようにしつつ，より低次元へと変換することである．

高次元のデータは，可視化やデータの特徴の理解が難しいだけでなく，計算コストが高くなる，統計的な推定が不安定になるといった問題を引き起こす（いわゆる「次元の呪い」）．
そのため，機械学習においては，次元削減は重要な前処理技術である．
次元削減の主な目的には次のものがある：
- 構造の抽出: データが本来持つ「低次元の本質的構造」を明らかにする
- ノイズ除去: 重要な成分だけを残して不要な変動（ノイズ）を除去する
- 可視化: 高次元のデータを，人間が理解しやすい2次元や3次元に投影する
- 計算効率の向上: 高次元のデータを扱う学習器の処理時間やメモリ使用量を削減する

以下では，代表的な線形手法である「主成分分析（PCA）」を紹介する．次回以降，非線形な構造を扱うための手法についても触れる．

---
### 主成分分析

統計手法である **主成分分析** (Principal Component Analysis, PCA)では，次元削減の変換を線形変換によって構成する手法の一つである．

以下では，$N$個の$D$次元ベクトルから成る学習データ

$$
\{ \pmb{x}_n \in \mathbb{R}^{D} | n = 1, 2, \ldots, N\}
$$

が与えられるとして，これを $H(\leq D)$ 次元に次元削減する場合を考える．また，話を簡単にするために，学習データの平均は $\pmb{0}$ である，すなわち

$$
\frac{1}{N}\sum_{n=1}^{N}\pmb{x}_{n} = \pmb{0}
$$

と仮定する（$\pmb{0}$でない場合の扱いについては後述する）．

#### 削減後の次元数が1の場合

---
［主成分分析の問題設定（1次元にする場合）］

上で説明したような学習データが与えられるとする．
このとき，大きさ $1$ の $D$ 次元ベクトル $\pmb{u}$ ($\Vert\pmb{u}\Vert = 1$)を用いて，次式によって $D$次元ベクトル $\pmb{x}$ を1つの実数値 $z_n$ に変換することを考える．

$$
z_n = \pmb{u}\cdot \pmb{x}_n \quad (n = 1, 2, \ldots, N)
$$

ベクトル $\pmb{u}$ を，大きさ1のベクトルの中で $\{ z_n \}$の分散が最大となるように定めたい．

---

この問題の解，つまり，$z_n$ の分散を最大にする $\pmb{u}$ は，「**データ$\pmb{x}_n$の分散共分散行列の固有ベクトルのうち，最大の固有値に対応するもの**」となる（導出過程は後述）．
したがって，学習データから分散共分散行列を計算し，その固有値固有ベクトルを求めれば，主成分分析による次元削減を実現できる．




上記の解の導出過程の概略を以下に示す．

まず，$\pmb{x}_n$ の平均が $\pmb{0}$ なので，$z_n$ の平均も $0$ となる（Q1）．そのため，$z_n$ の分散は

$$
\frac{1}{N}\sum_{n=1}^{N}z_n^2
$$

と表せる．このとき，

$$
\frac{1}{N}\sum_{n=1}^{N}z_n^2 = \pmb{u}^{\top}V\pmb{u}
$$

と書ける（Q2）．ただし，$V$ は学習データの分散共分散行列である．



ここで求めたいのは，$\{ z_n \}$ の分散 $\pmb{u}^\top V\pmb{u}$ を最大にするベクトル $\pmb{u}$ である．ただし，$\pmb{u}$ には $\Vert\pmb{u}\Vert = 1$ という条件がある．このような制約条件付きの最適化問題を解く定番の手法は，「ラグランジュの未定乗数法」である（注1）．
この問題の場合，ラグランジュ乗数を $\lambda$ として，

$$
L = \pmb{u}^{\top}V\pmb{u} - \lambda(\pmb{u}^{\top}\pmb{u} - 1)
$$

とおけば，$\frac{\partial L}{\partial \pmb{u}} = \pmb{0}$ を満たす $\pmb{u}$ が解の候補となる．


ここで，

$$
\frac{\partial L}{\partial \pmb{u}} = 2V\pmb{u} - 2\lambda\pmb{u}
$$

となる（注2）ので，解は

$$
V\pmb{u} = \lambda\pmb{u}
$$

を満たさねばならない．この式より，解の候補は $V$ の単位固有ベクトルであることがわかる．
この式を $L$ の式に代入すると，

$$
L = \lambda\pmb{u}^\top\pmb{u} - \lambda\cdot 0 = \lambda
$$

となるので，この問題の解は，$V$の最大固有値に対する単位固有ベクトルであることが分かる．


<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※注1: 「ラグランジュの未定乗数法」は，大学初年次の微積分で学んでいる...かも．興味のあるひとは数学の参考書を調べてね．</br>
※注2: $\frac{\partial L}{\partial \pmb{u}}$ は，「$L$を $\pmb{u}$ の各要素で偏微分したものをならべたベクトル」です．
</span>

**★★★ やってみよう ★★★**

Q1, Q2 を証明しよう．

#### 削減後の次元数が2以上の場合

主成分分析の目的は，線形変換によって，「元のデータの分散をなるべく保つような」次元削減を行うことである（厳密な問題の定式化は省略する）．

$D$ 次元ベクトル（$D\times 1 $ 行列） $\pmb{x}$ を $H$ 次元ベクトル（$H\times 1$行列） $\pmb{z}$  に線形変換する式を

$$
\pmb{z} = W\pmb{x}
$$

と表すと，この式は

$$
\pmb{z} = \begin{pmatrix}
z_1\\
z_2\\
\vdots\\
z_H
\end{pmatrix} = \begin{pmatrix}
\pmb{w}_1\cdot\pmb{x}\\
\pmb{w}_2\cdot\pmb{x}\\
\vdots\\
\pmb{w}_H\cdot\pmb{x}
\end{pmatrix}
$$

と書ける．このような線形変換 $W$ を定めることは，$H$個のベクトル $\pmb{w}_1, \pmb{w}_2,\ldots,\pmb{w}_H$ を定めることと同じである．

このとき，「元のデータの分散を保つ」という意味で最適な線形変換 $W$ は，次のようになる．



［主成分分析の問題の解］

$N$個の$D$次元ベクトルから成る学習データが与えられるとする．
$$
\{ \pmb{x}_n \in \mathbb{R}^{D} | n = 1, 2, \ldots, N\}
$$
ただし，このデータの平均は $\pmb{0}$ とする．
また，$\pmb{x}$ の分散共分散行列 $V = \frac{1}{N}\sum_{n=1}^{N}\pmb{x}_n\pmb{x}_n^{\top}$ の固有値を $\lambda_1 \geq \lambda_2 \geq \ldots \geq \lambda_D$とし，これらに対応する単位固有ベクトルをそれぞれ $\pmb{u}_1, \pmb{u}_2,\ldots,\pmb{u}_D$ とおく．

$H\times D$行列 $W$ によって

$$
\pmb{z} = W\pmb{x} = \begin{pmatrix}
\pmb{w}_1\cdot\pmb{x}\\
\pmb{w}_2\cdot\pmb{x}\\
\vdots\\
\pmb{w}_H\cdot\pmb{x}
\end{pmatrix}
$$

と次元削減するとき，主成分分析の目的を実現するためには，
ベクトル $\pmb{w}_h$ ($h = 1, 2, \ldots, H$) を $\pmb{u}_h$ の向きにとればよい（注）．

ここで，$\lambda_1$ から $\lambda_H$ （$H \leq D$）に対応する固有ベクトル $\pmb{u}_1, \pmb{u}_2, \ldots, \pmb{u}_H$ をならべた，次のような行列 $U_H$ を考える．

$$
U_H = (\pmb{u}_1\  \pmb{u}_2\  \ldots\ \pmb{u}_H )
$$

$U_H$ は $D \times H$ 行列である．このとき，主成分分析の変換行列 $W$ は，

$$
W = U_H^{\top} = \begin{pmatrix} \pmb{u}_1^{\top} \\ \pmb{u}_2^{\top} \\ \vdots \\ \pmb{u}_H^{\top} \end{pmatrix}
$$

となる．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※ 注: ここで考えている固有ベクトルは「単位」固有ベクトル（大きさが1，すなわち $||\pmb{u}_h||=1$）に限定しているが，符号の不定性がある（$\pmb{u}_h$も$−\pmb{u}_h$も固有ベクトル）．したがって，$\pmb{w}_h = -\pmb{u}_h$ としてもよい．
</span>

<img src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/ML/pcafig06b.png">

［データの平均が $\pmb{0}$ でない場合］

学習データ $\{ \pmb{x}_n \}$ の平均が $\pmb{0}$ でない場合の主成分分析を考える．
この場合，データの平均を $\pmb{\mu}$ として，つまり

$$
\pmb{\mu} = \frac{1}{N}\sum_{n=1}^{N}\pmb{x}_n
$$

として，$\pmb{x}'_n = \pmb{x}_n - \pmb{\mu}$ という変数変換を考えると，
$\pmb{x}'_n$ の平均は $\pmb{0}$ である．また，その分散共分散行列 $V$ は

$$
V = \frac{1}{N}\sum_{n=1}^{N} \pmb{x}'_n{\pmb{x}'_n}^{\top} = \frac{1}{N}\sum_{n=1}^{N} (\pmb{x}_n - \pmb{\mu})({\pmb{x}_n} - \pmb{\mu})^{\top}
$$

となり，$\pmb{x}_n$ の分散共分散行列に等しい．したがって，$V$ の固有値と固有ベクトルを求めて，次元削減の変換を次のようにすればよい．

$$
\pmb{z} = U_H^{\top}(\pmb{x} - \pmb{\mu}) = \begin{pmatrix}
\pmb{u}_1\cdot(\pmb{x} - \pmb{\mu})\\
\pmb{u}_2\cdot(\pmb{x} - \pmb{\mu})\\
\vdots\\
\pmb{u}_H\cdot(\pmb{x} - \pmb{\mu})
\end{pmatrix}
$$



#### 変換後のデータの平均と分散

学習データ $\pmb{x}_n$ の平均が $\pmb{\mu}$ のとき，変換後のデータ $\pmb{z}_n$ の平均を $\pmb{\mu}_z$ とおくと，

$$
\begin{aligned}
\pmb{\mu}_z &= \frac{1}{N}\sum_{n=1}^{N}\pmb{z}_n = \frac{1}{N}\sum_{n=1}^{N} U_H^{\top}(\pmb{x}_n - \pmb{\mu}) = U_H^{\top}\left( \frac{1}{N}\sum_{n=1}^{N} (\pmb{x}_n - \pmb{\mu})\right)\\
&= U_H^{\top}\left( \frac{1}{N}\sum_{n=1}^{N}\pmb{x}_n - \frac{1}{N}\sum_{n=1}^{N}\pmb{\mu} \right) = U_H^{\top} \left( \pmb{\mu} - \pmb{\mu}\right) = \pmb{0}
\end{aligned}
$$

となる．

学習データの分散共分散行列を $V$ とし，変換後のデータの分散共分散行列を $V_z$ とおくと，

$$
\begin{aligned}
V_z &= \frac{1}{N}\sum_{n=1}^{N}(\pmb{z}_n - \pmb{\mu}_z)(\pmb{z}_n - \pmb{\mu}_z)^{\top} = \frac{1}{N}\sum_{n=1}^{N}\pmb{z}_n\pmb{z}_n^{\top}\\
&= \frac{1}{N}\sum_{n=1}^{N} U_H^{\top}(\pmb{x}_n - \pmb{\mu}) \left(U_H^{\top}(\pmb{x}_n - \pmb{\mu})\right)^{\top}
= \frac{1}{N}\sum_{n=1}^{N} U_H^{\top}(\pmb{x}_n - \pmb{\mu}) (\pmb{x}_n - \pmb{\mu})^{\top} U_H\\
&= U_H^{\top} V U_H  = \textrm{diag}(\lambda_1, \lambda_2, \ldots, \lambda_H)
\end{aligned}
$$

となる．したがって，変換後のデータの変数間の共分散は（相関も） $0$ である．
また，変換後のデータの $h$ 番目の変数 $y_h$ の分散は，$V$ の $h$ 番目の固有値 $\lambda_h$ に等しい．

#### 再構成とその誤差

$D$ 次元のデータ $\pmb{x}$ に対して $\pmb{z} = U_H^{\top}(\pmb{x} - \pmb{\mu})$ は $H$ 次元である．このとき，

$$
\widehat{\pmb{x}} = U_H\pmb{z} + \pmb{\mu} = U_HU_H^{\top}(\pmb{x} -  \pmb{\mu}) + \pmb{\mu}
$$

は $D$ 次元である．これは，$\pmb{x}$ を $H$ 次元に次元削減してから元の次元数に戻したものになっており，「再構成」(reconstruction)と呼ばれる．

主成分分析は，学習データとその再構成との間の二乗誤差

$$
\sum_{n=1}^{N} \Vert\pmb{x}_n - \widehat{\pmb{x}}_n \Vert^2
$$

を最小化する線形変換となっている．つまり，任意の $H\times D$ 行列 $A$ と $D \times H$ 行列 $B$ を用いて $\widehat{\pmb{x}} = BA(\pmb{x} - \pmb{\mu}) + \pmb{\mu}$ と再構成することを考えたとき，$A = U_H^{\top}, B = U_H$ とすると再構成の二乗誤差が最小となる（注）．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※ 注: 「最小になるのは $A = U_H^{\top}, B = U_H$ のときに限られる」とは言ってないことに注意．実際，任意の $H\times H$ 正則行列 $C$ に対して $A = CU_H^{\top}, B = U_H C^{-1}$ とおけば上記の二乗誤差は最小となる．
</span>

#### 実験: 3次元データの次元削減

「数学」「物理」「情報」の点数のデータに主成分分析を適用してみる．

In [ ]:
# 数物情データを入手
! wget -nc https://www-tlab.math.ryukoku.ac.jp/~takataka/course/PIP/mpi100-mac.csv
dfMPI = pd.read_csv('mpi100-mac.csv', index_col=0)
datMPI = dfMPI.to_numpy().astype(float)
dfMPI

データの平均を求め，平均を引いたものを変数 `X` に格納し，データの分散共分散行列を求める．

In [ ]:
# 平均を求める
Xm = np.mean(datMPI, axis=0)
print('平均:', Xm)
print()

# 平均を引いて NumPy 配列にする
X = dfMPI.to_numpy() - Xm
N, D = X.shape
print('X.shape:', X.shape) # X は 100 x 3
print(X[:5, :]) # 最初の5人分を表示
print()

# 分散共分散行列
V = X.T @ X / N
print('V.shape:', V.shape)
print(V)

`X` のペアプロットを描くと下図のようになる．

In [ ]:
# X のペアプロット
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
L = [(0, 1), (0, 2), (1, 2)]
for i, (a, b) in enumerate(L):
    ax[i].scatter(X[:, a], X[:, b], s=8)
    ax[i].axvline(0, linestyle='-', color='gray')
    ax[i].axhline(0, linestyle='-', color='gray')
    ax[i].set_xlim(-40, 40)
    ax[i].set_ylim(-40, 40)
    ax[i].set_aspect('equal')
    ax[i].set_xlabel(f'$x_{a+1}$')
    ax[i].set_ylabel(f'$x_{b+1}$')
plt.tight_layout()
plt.show()

以下のセルを実行すると，`X` の分散共分散行列の固有値と固有ベクトルが求まる．
ここでは，特異値分解（注）という手法を使ってそれらを求めているが，この手法の説明は省略する．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※ 注: 特異値分解（SVD, Singular Value Decomposition）
</span>

In [ ]:
# 分散共分散行列の固有値と固有ベクトルを求める（Xの特異値分解経由で）
_, sval, Vt = np.linalg.svd(X, full_matrices=False)
eval = sval**2/N
U = Vt
for d in range(D):
    print(f'{d+1}番目の固有値:{eval[d]:.2f}   固有ベクトル:', U[d, :])

得られた3つの固有ベクトルを並べた行列 `U` を用いて `X` を変換したものを `Y` とする（次元数は変わらない）．さらに，その分散共分散行列を求める．

In [ ]:
# 変換後のデータ
Y = X @ U.T
print('Y.shape', Y.shape)
print(Y[:5, :]) # 最初の5人分を表示
print()

# 変換後の分散共分散行列
Vy = Y.T @ Y / N
print('Vy.shape:', Vy.shape)
print(Vy)

変換後の分散共分散行列 `Vy` は対角行列で，対角には `V` の固有値が大きい方から順に並んでいることが分かる．

また，変換後のデータのペアプロットは次のようになる．

In [ ]:
# Y のペアプロット
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
L = [(0, 1), (0, 2), (1, 2)]
for i, (a, b) in enumerate(L):
    ax[i].scatter(Y[:, a], Y[:, b], s=8)
    ax[i].axvline(0, linestyle='-', color='gray')
    ax[i].axhline(0, linestyle='-', color='gray')
    ax[i].set_xlim(-40, 40)
    ax[i].set_ylim(-40, 40)
    ax[i].set_aspect('equal')
    ax[i].set_xlabel(f'$y_{a+1}$')
    ax[i].set_ylabel(f'$y_{b+1}$')
plt.tight_layout()
plt.show()

`Vy` が対角行列であり対角要素が降順に並んでいる，ということから予想される通りの散布図になっていることが分かる．

次に，再構成して二乗誤差を測る実験を行ってみる．

In [ ]:
# Y から元データを再構成
print('Y.shape', Y.shape)
Z = Y @ U + Xm
print('Z.shape', Z.shape)
print(Z[:5, :]) # 最初の5人分を表示
# 再構成誤差を計算
msqe = np.sum((datMPI - Z)**2) / N
print(f'mean squared error = {msqe:.3f}')

次元削減しないで変換しただけのこの場合，再構成の誤差は 0 で，完全に元通りになっている．

次のセルを実行すると，2次元に次元削減→再構成と，1次元に次元削減→再構成の実験を行う．

In [ ]:
print('##### 2次元に次元削減してから再構成 #####')
Y2 = X @ U[:2, :].T
print('Y2.shape', Y2.shape)
Z2 = Y2 @ U[:2, :] + Xm
print('Z2.shape', Z2.shape)
print(Z2[:5, :]) # 最初の5人分を表示
msqe2 = np.sum((datMPI - Z2)**2) / N
print(f'mean squared error = {msqe2:.3f}')
print()

print('##### 1次元に次元削減してから再構成 #####')
Y1 = X @ U[:1, :].T
print('Y1.shape', Y1.shape)
Z1 = Y1 @ U[:1, :] + Xm
print('Z1.shape', Z1.shape)
print(Z1[:5, :]) # 最初の5人分を表示
msqe1 = np.sum((datMPI - Z1)**2) / N
print(f'mean squared error = {msqe1:.3f}')